# WFile Asia Public Notice Dataset

A quick exploration of multilingual public-information coverage from **WFile**.

- Website: https://wfile.org
- Dataset DOI: https://doi.org/10.5281/zenodo.24503671
- Dataset: `eranewsnew/wfile-asia-public-notice-dataset`
- Metadata only; article text remains subject to the original publishers' terms.


In [ ]:
import json, os, zipfile, urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

INPUT_ROOT = Path('/kaggle/input')
dataset_dir = None
if INPUT_ROOT.exists():
    for candidate in INPUT_ROOT.iterdir():
        if candidate.is_dir() and ((candidate / 'data.zip').exists() or (candidate / 'data' / 'articles.csv').exists()):
            dataset_dir = candidate
            break
print('Dataset dir:', dataset_dir)
work = Path('/kaggle/working/wfile-data')
work.mkdir(parents=True, exist_ok=True)
if dataset_dir and (dataset_dir / 'data.zip').exists():
    with zipfile.ZipFile(dataset_dir / 'data.zip') as z:
        z.extractall(work)
elif dataset_dir and (dataset_dir / 'data' / 'articles.csv').exists():
    for name in ['articles.csv','stats.json','sources.json']:
        src = dataset_dir / 'data' / name
        if src.exists():
            (work / 'data').mkdir(parents=True, exist_ok=True)
            (work / 'data' / name).write_bytes(src.read_bytes())
else:
    # Fallback keeps the notebook runnable if Kaggle has not mounted the dataset yet.
    (work / 'data').mkdir(parents=True, exist_ok=True)
    for name in ['articles.csv','stats.json','sources.json']:
        urllib.request.urlretrieve(f'https://wfile.org/data/{name}', work / 'data' / name)
print('Extracted:', sorted(p.name for p in work.iterdir()))

In [ ]:
articles = pd.read_csv(work / 'data/articles.csv')
stats = json.load(open(work / 'data/stats.json'))
print('Rows:', len(articles))
print('Stories:', stats.get('stories'), '| Languages:', stats.get('languages'))
print('Regions:', stats.get('regions'))
articles[['lang','region','section','title']].head()

In [ ]:
lang_counts = articles['lang'].value_counts()
lang_counts.plot(kind='bar', title='Published language versions', figsize=(8,3))
plt.ylabel('Articles')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
articles['published_at'] = pd.to_datetime(articles['published_at'], errors='coerce')
daily = articles.dropna(subset=['published_at']).groupby(articles['published_at'].dt.date).size()
daily.tail(14).plot(kind='line', marker='o', title='Published versions by day', figsize=(9,3))
plt.ylabel('Versions')
plt.tight_layout()
plt.show()
print('Date range:', articles['published_at'].min(), 'to', articles['published_at'].max())

In [ ]:
def search(term: str, lang: str = 'All', region: str = 'All', limit: int = 20):
    view = articles.copy()
    if term:
        mask = view['title'].fillna('').str.contains(term, case=False, regex=False) | view['summary'].fillna('').str.contains(term, case=False, regex=False)
        view = view[mask]
    if lang != 'All':
        view = view[view['lang'] == lang]
    if region != 'All':
        view = view[view['region'] == region]
    cols = ['lang','region','section','title','url','published_at']
    return view[cols].head(limit)

search('Osaka', limit=10)

## Links

- WFile: https://wfile.org
- Dataset DOI: https://doi.org/10.5281/zenodo.24503671
- GitHub mirror: https://github.com/adnxx1wsx/wfile-open-data
- Hugging Face mirror: https://huggingface.co/datasets/beijizhidayww/wfile-asia-public-notice-dataset
